# Legacy ACEF Full-BMAD phase analysis

This notebook reconstructs the frozen `P0-legacy-r2` timing profile without rerunning ACEF. It reads the pilot record plus the recorded parent and child Codex sessions. Active `task_started` → `task_complete` turns are measured, and concurrent turns are divided equally so allocations sum to elapsed wall time.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import pandas as pd

analysis_path = Path.cwd() / 'docs/experiments/execution-assurance-v2/legacy_phase_analysis.py'
spec = spec_from_file_location('legacy_phase_analysis', analysis_path)
analysis = module_from_spec(spec)
spec.loader.exec_module(analysis)
result = analysis.reconstruct()
result['run']

## Wall-time allocation

The table below is the decision view. `Conductor-only / orchestration` means no parent-visible worker was active; it can include inspection, commits, ledger updates, debugging, and dispatch coordination.

In [ ]:
stages = pd.DataFrame(result['stages'])
stages.assign(share_percent=(stages['share_of_timestamp_wall'] * 100).round(1))

## Detailed control categories

Category minutes are overlap-adjusted wall-clock allocations, not a sum of worker spans.

In [ ]:
categories = pd.DataFrame(result['categories'])
categories.assign(share_percent=(categories['share_of_timestamp_wall'] * 100).round(1))

## Longest active actor turns and retry cost

In [ ]:
actors = pd.DataFrame(result['actors'])
display(actors.head(12))
result['timing']

## Test execution time

Recognizable test and product-audit batches are discovered recursively across parent-visible and nested child sessions.

In [ ]:
pd.DataFrame(result['test_runtime']['test']['categories']), result['test_runtime']

## Data quality

The analysis can attribute parent-visible worker intervals and conductor-only gaps. Nested sessions are inspected for recognizable test exec batches but are not separately allocated in the top-level phase timeline. Harness wait and the exact product-done timestamp were not recorded in this legacy row.

In [ ]:
result['quality']

## Event-level assurance timing

The assurance-only reconstruction reconciles 18 top-level actor turns and 500 tool events. Top-level buckets are mutually exclusive. Nested reviewer activity is reported separately because it runs inside the parent critical path and can overlap.

In [ ]:
assurance_path = Path.cwd() / 'docs/experiments/execution-assurance-v2/legacy_assurance_timing.py'
assurance_spec = spec_from_file_location('legacy_assurance_timing', assurance_path)
assurance_analysis = module_from_spec(assurance_spec)
assurance_spec.loader.exec_module(assurance_analysis)
assurance = assurance_analysis.reconstruct(include_events=True)
assurance['summary']

## Assurance buckets and residual location

In [ ]:
display(pd.DataFrame(assurance['buckets']))
pd.DataFrame(assurance['residual_phases'])

## Model-cycle and token diagnostics

Token snapshots are bounded to each actor's first active turn. Cached input is a subset of input tokens, and reasoning output is a subset of output tokens. Correlations are descriptive across 18 actors, not causal estimates.

In [ ]:
model_rows = pd.DataFrame([{**{k: a[k] for k in ['story', 'control', 'task', 'elapsed_minutes', 'model', 'effort', 'model_cycle_count']}, **a['token_usage']} for a in assurance['actors']])
display(pd.DataFrame(assurance['control_model_usage']))
model_rows[['elapsed_minutes', 'model_cycle_count', 'total_tokens']].corr()

## Exact actor and nested-reviewer detail

In [ ]:
actor_columns = ['story', 'control', 'task', 'elapsed_minutes', 'model', 'effort', 'model_cycle_count', 'tool_event_count', 'nested_reviewer_count', 'nested_active_minutes_non_additive']
display(pd.DataFrame(assurance['actors'])[actor_columns])
pd.DataFrame(assurance['nested_reviewers'])

## Longest uninstrumented gaps

These are intervals with no live observed tool call. They cannot distinguish model inference from harness or queue latency.

In [ ]:
pd.DataFrame(assurance['residual_gaps']).head(25)